# 9. Triplet validation agreement

Compute inter-annotator agreement on triplet accept/reject verdicts from the
anonymized review log [triplet_reviews.jsonl](../protocol/inputs/triplet_reviews.jsonl):
one row per verdict, pseudonymous annotators, reviews ordered within each item
and validation round, restricted to the released items. Severity and QA
agreement live in notebooks 6-7 and
[qa_agreement.json](../protocol/results/qa_agreement.json).

In [1]:
import json

from pandas import DataFrame

with open("../protocol/inputs/triplet_reviews.jsonl", encoding="utf-8") as handle:
    reviews = DataFrame(json.loads(line) for line in handle)


def agreement(pairs):
    """Two-rater nominal agreement metrics on one accepted/flagged pair per item."""
    wide = pairs.pivot(index="item_id", columns="review_order", values="verdict").dropna()
    observed = (wide[1] == wide[2]).mean()
    labels = list(wide[1]) + list(wide[2])
    accept = labels.count("accepted") / len(labels)
    count = len(labels)
    chance_kappa = accept**2 + (1 - accept) ** 2
    chance_alpha = 1 - (
        labels.count("accepted") * (labels.count("accepted") - 1)
        + labels.count("flagged") * (labels.count("flagged") - 1)
    ) / (count * (count - 1))
    chance_ac1 = 2 * accept * (1 - accept)
    return {
        "items with two reviews": len(wide),
        "raw agreement": round(observed, 3),
        "share of accepted verdicts": round(accept, 3),
        "Krippendorff alpha (nominal)": round(1 - (1 - observed) / chance_alpha, 3),
        "Gwet AC1": round((observed - chance_ac1) / (1 - chance_ac1), 3),
        "PABAK": round(2 * observed - 1, 3),
        "chance agreement assumed by kappa/alpha": round(chance_kappa, 3),
        "chance agreement assumed by AC1": round(chance_ac1, 3),
    }


table = DataFrame(
    {
        name: agreement(reviews[(reviews["round"] == name) & (reviews.review_order <= 2)])
        for name in ("round1", "round2")
    }
)
table

,round1,round2
items with two reviews,972.000,158.000
raw agreement,0.836,0.506
share of accepted verdicts,0.909,0.538
Krippendorff alpha (nominal),0.012,0.010
Gwet AC1,0.804,0.018
PABAK,0.673,0.013
chance agreement assumed by kappa/alpha,0.834,0.503
chance agreement assumed by AC1,0.166,0.497


## Reading the coefficients

First-round verdicts are heavily skewed toward acceptance, so the kappa/alpha
family assumes near-total chance agreement and collapses despite high raw
agreement: the prevalence paradox (Feinstein and Cicchetti 1990). Gwet's AC1
(Gwet 2008) keeps the chance term bounded under skew and is the skew-robust
statistic. On the balanced QA answer-choice task the same annotator pool
reaches nominal alpha 0.89, so the near-zero verdict alpha reflects the label
distribution, not the annotators.

The rounds are not comparable. A triplet received a second within-round review
in round 2 precisely when it was contested, so that subset is selected for
disagreement, and its balanced verdicts need no prevalence correction.

In [2]:
second_round = reviews[reviews["round"] == "round2"]
review_counts = second_round.groupby("item_id").review_order.max()
singles = second_round[second_round.item_id.isin(review_counts[review_counts == 1].index)]
paired = second_round[
    second_round.item_id.isin(review_counts[review_counts >= 2].index)
    & (second_round.review_order <= 2)
]
DataFrame(
    {
        "items": [singles.item_id.nunique(), paired.item_id.nunique()],
        "share of accepted verdicts": [
            round(singles.verdict.eq("accepted").mean(), 3),
            round(paired.verdict.eq("accepted").mean(), 3),
        ],
    },
    index=["closed by a single round-2 review", "two round-2 reviews (table above)"],
)

,items,share of accepted verdicts
closed by a single round-2 review,537,0.935
two round-2 reviews (table above),158,0.538
